# Entrenar en Kaggle (GPU) - copiloto de triaje, SP1

Hace lo mismo que `notebooks/entrenar_gpu.ipynb` hace en Colab. Cambia por tres
motivos de plataforma:

1. **Cuota.** Kaggle da 30 horas de GPU por semana con un contador propio,
   independiente del de Colab.
2. **El dataset es nativo.** RSNA se adjunta desde el propio Kaggle: no se
   descargan 3,7 GB ni hace falta la clave de la API. Llega **ya
   descomprimido** y de **solo lectura**, y por eso
   `scripts/preparar_datos.py` acepta `--crudo` además de `--zip`.
3. **El caché de 320 px.** La conversión DICOM -> PNG son unos 35 minutos que
   se pagaban enteros en cada sesión de Colab. Aquí se guardan los PNG **ya
   redimensionados** y `Save Version` los convierte en un Dataset reutilizable:
   la segunda corrida arranca sin conversión. La celda 7 explica y **mide** por
   qué redimensionar antes es equivalente a hacerlo al vuelo.

La lógica vive en el paquete `copiloto_triaje`, que tiene pruebas. Estas celdas
solo orquestan: eligen rutas, llaman y escriben los entregables. Es el mismo
reparto que exige el spec (párrafo 5: "el notebook no contiene lógica").

## Paso 0 - dejar la sesión lista (antes de ejecutar nada)

En este orden:

1. **Create > New Notebook** en kaggle.com.
2. Panel derecho, **Session options**:
   - **Accelerator**: `GPU T4 x2` o `GPU P100`. Sin esto la celda 3 se detiene.
   - **Internet**: `On`. Hace falta dos veces: para que `pip` instale el
     paquete y para que torchvision descargue los pesos de ImageNet.
3. **+ Add Input > Competitions**, buscar *RSNA Pneumonia Detection Challenge*
   y **Add**. Aparece montado en
   `/kaggle/input/rsna-pneumonia-detection-challenge/`. Requiere tener
   aceptadas las reglas de la competencia (ya lo están).
4. **Subir el código como Dataset privado** (vía principal; hay una
   alternativa en la celda 2):
   - En la carpeta del repo, con todo ya commiteado:
     `git archive --format=zip --prefix=copiloto-triaje/ -o ../copiloto-triaje.zip HEAD`
     El `git archive` se usa a propósito en vez de comprimir la carpeta desde
     el Explorador, que metería `.git`, `__pycache__` y `.pytest_cache`. El
     `--prefix` deja el árbol colgando de una carpeta, que es lo que busca la
     celda 2.
   - **Create > New Dataset**, subir ese zip (Kaggle lo descomprime solo),
     título `copiloto-triaje-codigo`, visibilidad **Private**, **Create**.
   - De vuelta en el cuaderno: **+ Add Input > Datasets > Your Datasets** y
     añadirlo. Cada vez que cambie el código hay que subir una **New Version**
     del Dataset.
5. **File > Import Notebook** y subir este `entrenar_kaggle.ipynb`.
6. Ejecutar las celdas en orden.

**Dos límites de Kaggle que condicionan cómo está escrito el resto:**

- Una sesión dura como máximo **12 horas** y la interactiva además se cierra
  sola al rato de no tocarla. Por eso todo entrena con `reanudar=True` y con
  los checkpoints en `/kaggle/working`, y por eso importa `Save Version`.
- Lo único escribible es **`/kaggle/working`** (unos 20 GB) y `/kaggle/temp`.
  `/kaggle/input` es de solo lectura, incluido el código.

In [ ]:
# Celda 1 - que maquina toco y que hay montado
# El assert de torch NO va aqui: la celda 2 instala el paquete y pip puede
# reemplazar el torch de la imagen de Kaggle para cumplir los pines de
# pyproject.toml. La comprobacion del entorno es la celda 3, ya con lo
# instalado y despues del posible reinicio.
!nvidia-smi
!df -h /kaggle/working

import os
import pathlib

entradas = pathlib.Path('/kaggle/input')
if entradas.is_dir():
    for entrada in sorted(entradas.iterdir()):
        print('input:', entrada.name)
else:
    print('AVISO: no hay /kaggle/input. Este cuaderno es para Kaggle.')
print('nucleos de CPU:', os.cpu_count())

## Celda 2 - instalar el paquete (no un `sys.path.insert`)

Se instala de verdad para que `pip` aplique los pines de `pyproject.toml`
(`torch==2.14.0`, `torchvision==0.29.0`, `pydicom==3.0.2`, ...). Con un
`sys.path.insert` el entrenamiento correría con las versiones que trae la
imagen de Kaggle y el entorno remoto dejaría de reproducir al local, que es lo
que el spec (párrafo 10) exige.

**Si `pip` sustituye el `torch` de la imagen**, el runtime sigue con el viejo
cargado en memoria: hay que ir al panel derecho > **Session options > Restart
session** (o menú **Run > Restart & clear cell outputs**) y volver a ejecutar
**desde la celda 3**; no hace falta repetir la instalación.

**Alternativa al Dataset privado**: subir el zip con el botón *Upload* del
panel de archivos (`/kaggle/working`) y descomprimirlo allí. Funciona, pero hay
que repetirlo en cada sesión, mientras que el Dataset queda adjunto.

In [ ]:
# Celda 2 - localizar el codigo e instalarlo
import pathlib
import shutil
import subprocess
import sys

# INSTALAR_DEPENDENCIAS=True aplica los pines de pyproject.toml: es lo que hace
# comparable esta corrida con la local, y cuesta una descarga grande mas un
# probable reinicio de sesion. Con False se instala solo el paquete (--no-deps)
# sobre las versiones de Kaggle: mas rapido, pero entonces la corrida NO es
# reproducible pin a pin y la celda 3 lo dice en voz alta en vez de callarselo.
INSTALAR_DEPENDENCIAS = True


def _caminar_podando(raiz, tope_archivos=100, profundidad_max=6):
    """Recorre el arbol saltandose las carpetas de datos.

    No se puede usar rglob a secas: los 26 684 DICOM de la competencia y los
    PNG de cualquier dataset procesado hacen que recorrerlo entero tarde
    minutos. Tampoco vale un limite fijo de niveles: Kaggle monta los Datasets
    como <input>/datasets/<owner>/<slug>/..., que ya son tres niveles antes de
    llegar al contenido, y ahi fue donde fallo el primer intento.

    La poda mira el NUMERO DE ARCHIVOS de cada carpeta: las de datos tienen
    miles y las de codigo unas decenas, asi que se deja de bajar en cuanto una
    carpeta parece un deposito de datos.
    """
    import os
    raiz = pathlib.Path(raiz)
    for actual, carpetas, archivos in os.walk(raiz):
        aqui = pathlib.Path(actual)
        nivel = len(aqui.relative_to(raiz).parts)
        if len(archivos) > tope_archivos or nivel >= profundidad_max:
            carpetas[:] = []      # no bajar mas: esto es un deposito de datos
        yield aqui, archivos


def buscar_repo(raiz=pathlib.Path('/kaggle/input')):
    """Devuelve (ajenas, propias): las carpetas con el codigo del proyecto.

    El slug del Dataset lo inventa Kaggle a partir del titulo, asi que
    escribirlo fijo aqui rompe el cuaderno en cuanto se titule de otra forma.
    Se busca por lo unico que no cambia: un pyproject.toml que declare este
    paquete.

    `propias` son las copias que dejo este mismo cuaderno en una sesion
    anterior (en /kaggle/working/codigo, que al adjuntar la salida aparece
    como <entrada>/codigo): pueden estar desactualizadas y por eso se separan.
    """
    ajenas, propias = [], []
    if not pathlib.Path(raiz).is_dir():
        return ajenas, propias
    for carpeta, archivos in _caminar_podando(raiz):
        if 'pyproject.toml' not in archivos:
            continue
        try:
            if 'copiloto-triaje' not in (carpeta / 'pyproject.toml').read_text(
                    encoding='utf-8'):
                continue
        except OSError:
            continue
        (propias if carpeta.name == 'codigo' else ajenas).append(carpeta)
    return sorted(ajenas), sorted(propias)


ajenas, propias = buscar_repo()
if not ajenas and propias:
    raise SystemExit(
        'Lo unico que encuentro es la copia que dejo una sesion anterior '
        f'({propias[0]}), no el Dataset del codigo. Puede estar '
        'desactualizada. Adjunta el Dataset del repo: + Add Input > Datasets.')
if not ajenas:
    # Un "no lo encuentro" a secas obliga a adivinar. Se lista lo que SI hay.
    lineas = ['No encuentro el codigo en /kaggle/input.', '',
              'Esto es lo que hay adjunto:']
    zips = []
    for entrada in sorted(pathlib.Path('/kaggle/input').iterdir()):
        if entrada.is_dir():
            hijos = sorted(h.name for h in entrada.iterdir())
            zips += [h for h in hijos if h.endswith('.zip')]
            lineas.append(f'  - {entrada.name}/   contiene: {hijos[:8]}')
        else:
            lineas.append(f'  - {entrada.name}   (archivo suelto)')
    if zips:
        lineas += ['', f'PISTA: hay un .zip sin descomprimir ({zips[0]}).',
                   'Kaggle no lo expande si se sube como archivo suelto:',
                   'vuelve a subirlo con + Add Input > Upload > New Dataset.']
    lineas += ['', 'Si no aparece: + Add Input > Upload > New Dataset, y',
               'suelta copiloto-triaje.zip.']
    raise SystemExit(chr(10).join(lineas))
assert len(ajenas) == 1, (
    'Hay mas de un Dataset con el codigo adjunto y no se cual quieres: '
    f'{[str(c) for c in ajenas]}. Desadjunta los que sobren.')
REPO = ajenas[0]
print('codigo encontrado en', REPO)
if propias:
    print('AVISO: se ignora la copia de una sesion anterior en', propias[0])

# Se copia a /kaggle/working antes de instalar porque pip construye el paquete
# DENTRO del arbol de fuentes (deja build/ y *.egg-info alli) y /kaggle/input
# es de solo lectura: instalar directo desde la entrada falla al escribir.
# Ademas deja registrado en la salida de la version exactamente que codigo
# corrio.
CODIGO = pathlib.Path('/kaggle/working/codigo')
if CODIGO.exists():
    shutil.rmtree(CODIGO)
shutil.copytree(REPO, CODIGO)
print('codigo copiado de', REPO, 'a', CODIGO)

orden = [sys.executable, '-m', 'pip', 'install', '-q', str(CODIGO)]
if not INSTALAR_DEPENDENCIAS:
    orden.insert(-1, '--no-deps')
# check=True y no `!pip install`: con la magia de shell, un fallo de pip se
# imprime y la celda sigue como si nada, y el error real aparece tres celdas
# mas abajo como un ImportError que no explica nada.
subprocess.run(orden, check=True)
print('paquete instalado')

In [ ]:
# Celda 3 - verificar el entorno DESPUES de instalar
import pathlib
import tomllib

# CODIGO se recalcula aqui en vez de heredarse de la celda 2: el reinicio de
# sesion que esta misma celda recomienda borra el estado de Python, y sin esto
# reejecutar "desde esta celda" -lo que dice el mensaje del assert- reventaba
# con NameError y obligaba a repetir la instalacion entera.
CODIGO = pathlib.Path('/kaggle/working/codigo')
assert (CODIGO / 'pyproject.toml').is_file(), (
    f'No esta {CODIGO}: ejecuta la celda 2 antes que esta.')

import torch
import torchvision

import copiloto_triaje

print('torch', torch.__version__, '| cuda', torch.version.cuda,
      '| torchvision', torchvision.__version__)
print('copiloto_triaje desde', copiloto_triaje.__file__)
# Lo que importa es que NO venga del arbol de fuentes, no que la carpeta se
# llame de una forma concreta: el Python de Debian -la base de la imagen de
# Kaggle- instala en dist-packages, no en site-packages, y comprobar el
# nombre daba un falso fallo con el paquete correctamente instalado.
ruta_modulo = pathlib.Path(copiloto_triaje.__file__).resolve()
for prohibida, motivo in ((CODIGO, 'la copia de las fuentes'),
                          (pathlib.Path('/kaggle/input'), 'un Dataset adjunto')):
    # Se resuelven los DOS lados: comparar una ruta resuelta con otra sin
    # resolver no falla nunca en el caso que importa.
    assert not ruta_modulo.is_relative_to(prohibida.resolve()), (
        f'copiloto_triaje se esta importando de {motivo} ({ruta_modulo}), no '
        'del paquete instalado: los pines de pyproject.toml no se aplicaron. '
        'Reejecuta la celda 2.')
print('OK: viene del paquete instalado, no del arbol de fuentes')
assert torch.cuda.is_available(), (
    'Sin GPU. Si acabas de instalar y pip reemplazo torch, reinicia la sesion '
    '(panel derecho > Session options > Restart session) y vuelve a ejecutar '
    'DESDE ESTA CELDA. Si no, panel derecho > Session options > Accelerator > '
    'GPU T4 x2.')
print('GPU:', torch.cuda.get_device_name(0))

# Los pines se leen del pyproject.toml que se acaba de instalar en vez de
# repetirlos aqui: una copia a mano se desincroniza en silencio y entonces
# esta comprobacion daria por bueno un entorno que ya no corresponde.
pines = tomllib.loads(
    (CODIGO / 'pyproject.toml').read_text(encoding='utf-8'))['project']['dependencies']
fijados = dict(d.split('==') for d in pines if '==' in d)
for paquete, instalada in (('torch', torch.__version__),
                           ('torchvision', torchvision.__version__)):
    # torch informa '2.14.0+cu130': la etiqueta de compilacion CUDA depende de
    # la imagen de Kaggle y no es parte del pin, asi que se compara lo anterior
    # al '+'.
    real = instalada.split('+')[0]
    esperada = fijados.get(paquete)
    if esperada and real != esperada:
        print(f'AVISO: {paquete} {real} no es el pin {esperada}. La corrida '
              f'sigue siendo valida, pero NO es comparable pin a pin con la '
              f'local: anotalo junto a las metricas.')
    else:
        print(f'OK {paquete} {real} (pin {esperada})')

## Celda 4 - rutas, y el caché de una corrida anterior

Tres cosas pueden venir ya hechas de una sesión anterior, si se adjuntó la
salida de esa versión con **+ Add Input > Your Work / Notebook Output**:

- `png320/`: los PNG ya redimensionados. Ahorra los ~35 minutos de conversión.
- `trabajo/manifiestos/`: la partición por paciente. **Tiene que venir con el
  caché**, porque la partición depende de la semilla y de qué imágenes se
  convirtieron: rehacerla por separado podría repartir a los pacientes de otra
  forma y contaminar la comparación entre corridas.
- `corridas/`: los checkpoints. Se copian a `/kaggle/working` porque reanudar
  escribe `ultimo.pt` en esa misma carpeta y `/kaggle/input` es de solo
  lectura.

In [ ]:
# Celda 4 - rutas de trabajo y reutilizacion de una salida anterior
import pathlib
import shutil

from PIL import Image

def buscar_competencia(raiz=pathlib.Path('/kaggle/input')):
    """Localiza la carpeta de la competencia por su contenido, no por su ruta.

    La ruta fija /kaggle/input/rsna-pneumonia-detection-challenge solo vale
    cuando Kaggle monta la competencia en la raiz. Con el formato nuevo queda
    en <input>/competitions/<slug>/, y al adjuntar por kagglehub en
    <input>/datasets/<owner>/<slug>/. Se busca lo unico que la identifica de
    verdad: que contenga stage_2_train_labels.csv y stage_2_train_images/.

    Esto ademas distingue la competencia OFICIAL de los datasets procesados
    por terceros que tambien se llaman "rsna-pneumonia-algo": esos no traen
    esos dos nombres, y entrenar con la version de otro sin saber como la
    proceso invalidaria la comparacion con la corrida local.
    """
    import os
    raiz = pathlib.Path(raiz)
    if not raiz.is_dir():
        return None
    for actual, carpetas, archivos in os.walk(raiz):
        aqui = pathlib.Path(actual)
        if len(archivos) > 100 or len(aqui.relative_to(raiz).parts) >= 6:
            carpetas[:] = []
        if 'stage_2_train_labels.csv' in archivos and 'stage_2_train_images' in carpetas:
            return aqui
    return None


CRUDO = buscar_competencia()
TRABAJO = pathlib.Path('/kaggle/working/trabajo')
PNG320 = pathlib.Path('/kaggle/working/png320')
CORRIDAS = pathlib.Path('/kaggle/working/corridas')

# El cache se construye al tamano con el que se entrena. Cambiar uno sin el
# otro es un error silencioso (ver la celda 7), asi que sale de una sola
# constante.
TAM_CACHE = 320

# LIMITE > 0 procesa solo N imagenes: sirve para recorrer el cuaderno entero en
# unos minutos y comprobar que no se rompe nada ANTES de gastar cuota de GPU.
# Con LIMITE > 0 las metricas no valen para nada, solo la mecanica.
LIMITE = 0

if CRUDO is None:
    disponibles = []
    for entrada in sorted(pathlib.Path('/kaggle/input').iterdir()):
        disponibles.append(f'  - {entrada.name}')
    raise SystemExit(chr(10).join(
        ['No encuentro la competencia: ninguna carpeta adjunta trae',
         'stage_2_train_labels.csv junto a stage_2_train_images/.', '',
         'Adjunto ahora mismo:'] + disponibles +
        ['', 'Adjuntala con: + Add Input > Competitions > RSNA Pneumonia',
         'Detection Challenge (hay que tener aceptadas sus reglas).', '',
         'OJO: un dataset de un tercero llamado "rsna-pneumonia-algo" NO',
         'sirve aqui. No se sabe como lo proceso, asi que sus metricas no',
         'serian comparables con la corrida local, que es el objetivo.']))
print('competencia encontrada en', CRUDO)


def buscar_en_entradas(relativa, raiz=pathlib.Path('/kaggle/input')):
    """Busca <entrada>/<relativa> en los Datasets adjuntos.

    Al adjuntar la salida de una version, Kaggle monta lo que habia en
    /kaggle/working colgando del nombre de esa entrada, de modo que las rutas
    relativas se conservan y basta con probarlas una a una.
    """
    if not raiz.is_dir():
        return None
    for entrada in sorted(raiz.iterdir()):
        candidata = entrada / relativa
        if candidata.is_dir():
            return candidata
    return None


cache_png = buscar_en_entradas('png320')
cache_manifiestos = buscar_en_entradas('trabajo/manifiestos')
HAY_CACHE = cache_png is not None and cache_manifiestos is not None

if HAY_CACHE:
    # Los PNG se leen directamente desde /kaggle/input: son de solo lectura,
    # pero aqui solo se leen, y copiar medio giga a working no aporta nada.
    PNG320 = cache_png
    # Los manifiestos SI se copian: el resto del cuaderno los busca por
    # cfg.dir_manifiestos, que cuelga de la raiz de trabajo.
    destino_manifiestos = TRABAJO / 'manifiestos'
    destino_manifiestos.mkdir(parents=True, exist_ok=True)
    for csv in cache_manifiestos.glob('*.csv'):
        shutil.copy2(csv, destino_manifiestos / csv.name)
    # Los recuentos de la preparacion viajan con los manifiestos: la celda 9 se
    # los pasa al reporte, y sin ellos los reportes de una sesion con cache
    # pierden en silencio la seccion de archivos omitidos, que es entregable.
    for nombre in ('omitidos.json', 'sin_etiqueta.json', 'descartados_por_edad.json'):
        origen_recuento = cache_manifiestos.parent / nombre
        if origen_recuento.is_file():
            shutil.copy2(origen_recuento, TRABAJO / nombre)
    # Comprobar la resolucion REAL de los PNG adjuntos, no TAM_CACHE contra si
    # mismo. Sin esto, adjuntar un cache de 320 px con TAM_CACHE=224 -algo que
    # el propio cuaderno presenta como posible- pasaba todos los controles y
    # redimensionaba las 25 403 imagenes por segunda vez en silencio, con un
    # reporte que declaraba 224.
    primera = next(iter(sorted(PNG320.glob('*.png'))), None)
    assert primera is not None, f'El cache adjunto {PNG320} no tiene ningun PNG.'
    with Image.open(primera) as archivo:
        medida = archivo.size
    assert medida == (TAM_CACHE, TAM_CACHE), (
        f'El cache adjunto esta a {medida[0]} px y TAM_CACHE es {TAM_CACHE}. '
        'Entrenarias sobre imagenes redimensionadas dos veces. O pones '
        f'TAM_CACHE = {medida[0]}, o generas un cache nuevo a {TAM_CACHE} '
        'ejecutando las celdas 5 a 7 sin adjuntar el cache viejo.')
    print('cache adjunto:', PNG320, '| resolucion verificada:', medida)
    print('manifiestos copiados a', destino_manifiestos)
else:
    print('sin cache adjunto: se hara la conversion completa (unos 35 min)')

cache_corridas = buscar_en_entradas('corridas')
if cache_corridas is not None and not CORRIDAS.exists():
    shutil.copytree(cache_corridas, CORRIDAS)
    print('corridas anteriores restauradas desde', cache_corridas)
CORRIDAS.mkdir(parents=True, exist_ok=True)

## Celda 5 - preparar los datos

`--crudo` en vez de `--zip`: el dataset llega descomprimido y de solo lectura,
y el script lo lee sin escribir nada dentro.

Son unos 35 minutos. **Mira la salida**: si aparece una línea `AVISO`, hubo
imágenes convertidas que no entraron en los manifiestos, y eso es justo lo que
el spec (párrafo 8) exige que no pase en silencio.

In [ ]:
# Celda 5 - DICOM -> PNG -> manifiestos
import subprocess
import sys

if HAY_CACHE:
    print('cache de 320 px adjunto: no hay nada que convertir.')
else:
    # -u porque el hijo no escribe a una terminal: sin esto su stdout queda
    # con buffer de bloque y los [1/4]...[4/4] aparecen de golpe al terminar,
    # de modo que durante 35 minutos no se distingue "convirtiendo" de
    # "colgado". Es el mismo motivo por el que la celda 9 usa flush=True.
    orden = [sys.executable, '-u', str(CODIGO / 'scripts' / 'preparar_datos.py'),
             '--crudo', str(CRUDO), '--raiz', str(TRABAJO)]
    if LIMITE:
        orden += ['--limite', str(LIMITE)]
    # check=True por lo mismo que en la celda 2: con `!python ...` un fallo a
    # mitad de la conversion deja manifiestos incompletos y la celda siguiente
    # entrena con ellos sin enterarse.
    subprocess.run(orden, check=True)

## Celda 6 y 7 - el caché de 320 px, y por qué es equivalente

Guardar los PNG ya redimensionados a 320x320 da **exactamente el mismo tensor**
que guardarlos a 1024x1024 y redimensionar al vuelo. Es una afirmación fuerte,
así que aquí está el argumento y, debajo, la medición.

El argumento, leyendo `construir_transformaciones` en
`src/copiloto_triaje/datos/dataset.py`:

```python
pasos = [transforms.Resize((tam_imagen, tam_imagen))]
if entrenamiento:
    pasos.append(transforms.RandomAffine(degrees=7, translate=(0.05, 0.05)))
    pasos.append(transforms.ColorJitter(brightness=0.15, contrast=0.15))
```

1. El primer paso es un `Resize` a un tamaño fijo: un redimensionado simple,
   **sin recorte**. No hay `RandomResizedCrop` ni `CenterCrop`, que son los que
   harían que el resultado dependiera del tamaño de partida.
2. El aumento se añade **después**. Con caché o sin él, `RandomAffine` y
   `ColorJitter` reciben una imagen de 320x320 idéntica.
3. `Resize` a un tamaño que la imagen ya tiene es una operación nula: Pillow
   devuelve una copia. El PNG del caché atraviesa el `Compose` sin que lo
   toquen.
4. El caché se escribe con `Image.BILINEAR`, que es el filtro por defecto de
   `transforms.Resize`. No es un detalle: con otro filtro el caché dejaría de
   ser equivalente y nada lo avisaría.
5. El dataset hace `convert("RGB")` sobre el gris al cargar. Redimensionar en
   gris y replicar después da lo mismo que replicar y redimensionar, porque el
   filtro bilineal actúa canal a canal sobre tres canales idénticos.

Medido en local antes de escribir esto, sobre ruido y sobre un gradiente:
`max|A - B| = 0.0`, con y sin aumento. La celda 6 lo vuelve a medir **sobre una
radiografía real** del dataset para que no haya que creérselo.

**Lo que NO vale**: entrenar a 224 px desde este caché. Sería redimensionar dos
veces (1024 -> 320 -> 224), que no es lo mismo que 1024 -> 224. Si algún día se
entrena a otro tamaño, hay que regenerar el caché a ese tamaño.

El ahorro de espacio es geométrico: el área baja a 320^2 / 1024^2 = 9,8% de la
original. La celda 7 imprime el tamaño real que ocupó.

In [ ]:
# Celda 6 - comprobar la equivalencia sobre una imagen real
import pandas as pd
import torch
from PIL import Image

from copiloto_triaje.config import Config
from copiloto_triaje.datos.dataset import construir_transformaciones

cfg_rutas = Config(raiz_datos=TRABAJO, zip_rsna=CRUDO, tam_imagen=TAM_CACHE)
muestra = pd.read_csv(cfg_rutas.dir_manifiestos / 'train.csv').iloc[0]['archivo_png']
origen = cfg_rutas.dir_png / muestra

if not origen.exists():
    # Con el cache adjunto los PNG de 1024 px ya no estan (la celda 8 los
    # libera): no hay con que comparar, y saltar es lo correcto porque la
    # comprobacion ya se hizo en la sesion que construyo el cache.
    print('PNG de 1024 px no disponible (se esta usando el cache). '
          'La comprobacion corresponde a la sesion que lo genero.')
else:
    transformacion = construir_transformaciones(TAM_CACHE, entrenamiento=False)
    with Image.open(origen) as archivo:
        tensor_al_vuelo = transformacion(archivo.convert('RGB'))
    with Image.open(origen) as archivo:
        # EXACTAMENTE la expresion de reescribir() en la celda 7. Antes esta
        # llevaba un convert('L') que la celda 7 no hace: hoy da igual porque
        # datos/dicom.py escribe en modo L y convert('L') es un no-op, pero
        # era una comprobacion que habria seguido dando 0.0 si algun dia el
        # cache se escribiera en otro modo. Una comprobacion que pasa igual
        # con el cache roto no comprueba nada.
        reducida = archivo.resize((TAM_CACHE, TAM_CACHE), Image.BILINEAR)
    tensor_cacheado = transformacion(reducida.convert('RGB'))
    diferencia = float((tensor_al_vuelo - tensor_cacheado).abs().max())
    print(muestra, '| forma', tuple(tensor_al_vuelo.shape),
          '| max|al_vuelo - cacheado| =', diferencia)
    assert diferencia == 0.0, (
        'El cache NO es equivalente en esta version de Pillow/torchvision. No '
        'sigas: entrenarias sobre imagenes distintas a las que describe el '
        'reporte.')

In [ ]:
# Celda 7 - escribir el cache de 320 px
import os
import pathlib
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from PIL import Image

if HAY_CACHE:
    print('el cache ya viene adjunto en', PNG320)
else:
    PNG320.mkdir(parents=True, exist_ok=True)

    # Solo lo que aparece en los manifiestos. Los manifiestos ya excluyen a los
    # menores de 18 y las edades imposibles (filtro de edad de Config), asi que
    # reescribir el directorio entero meteria en el cache miles de imagenes que
    # ningun cargador va a pedir nunca.
    nombres = sorted({
        nombre
        for particion in ('train', 'val', 'test')
        for nombre in pd.read_csv(
            cfg_rutas.dir_manifiestos / f'{particion}.csv')['archivo_png']
    })

    def reescribir(nombre):
        destino = PNG320 / nombre
        if destino.exists():
            return False  # idempotente: reejecutar la celda no rehace el trabajo
        with Image.open(cfg_rutas.dir_png / nombre) as archivo:
            reducida = archivo.resize((TAM_CACHE, TAM_CACHE), Image.BILINEAR)
        # Escritura atomica, por lo mismo que en datos/dicom.py: un corte a
        # mitad del guardado deja un PNG truncado CON EL NOMBRE DEFINITIVO, y a
        # partir de ahi nadie lo distingue de uno bueno hasta que el
        # entrenamiento revienta horas despues.
        temporal = destino.with_name(destino.name + '.tmp')
        try:
            reducida.save(temporal, format='PNG')
            os.replace(temporal, destino)
        finally:
            temporal.unlink(missing_ok=True)
        return True

    # Hilos y no procesos: Pillow suelta el GIL al descodificar y codificar, y
    # en una maquina de pocos nucleos arrancar procesos y serializar imagenes
    # cuesta mas de lo que ahorra.
    inicio = time.time()
    hilos = min(8, (os.cpu_count() or 2) * 2)
    with ThreadPoolExecutor(max_workers=hilos) as grupo:
        escritos = sum(grupo.map(reescribir, nombres))
    segundos = time.time() - inicio
    print(f'{escritos} escritos, {len(nombres) - escritos} ya estaban '
          f'| {segundos / 60:.1f} min con {hilos} hilos')

bytes_cache = sum(f.stat().st_size for f in PNG320.glob('*.png'))
print(f'cache: {len(list(PNG320.glob("*.png")))} PNG | '
      f'{bytes_cache / 2**30:.2f} GB en {PNG320}')

## Celda 8 - liberar los PNG de 1024 px

`Save Version` convierte **todo** `/kaggle/working` en el Dataset de salida. Los
PNG de 1024 px son del orden de diez veces el caché y no sirven para entrenar:
el primer paso del `Compose` los reduce a 320 de todas formas. Se borran para
que la salida quepa holgada y para que adjuntarla en la próxima sesión sea
rápido.

Se borran **después** de comprobar que cada archivo de los manifiestos está en
el caché. Reconstruirlos cuesta otros 35 minutos desde `/kaggle/input`, así que
la pérdida es de tiempo, no de datos, pero no hay motivo para pagarla por un
descuido.

In [ ]:
# Celda 8 - borrar los PNG originales antes de Save Version
import shutil

LIBERAR_PNG_ORIGINALES = True

if HAY_CACHE:
    print('no hay PNG de 1024 px en esta sesion: nada que liberar')
elif not LIBERAR_PNG_ORIGINALES:
    print('conservando los PNG de 1024 px (ocuparan sitio en la salida)')
else:
    faltan = [n for n in nombres if not (PNG320 / n).exists()]
    assert not faltan, (
        f'{len(faltan)} imagen(es) de los manifiestos no estan en el cache, '
        f'la primera es {faltan[0]}. No se borra nada: vuelve a ejecutar la '
        f'celda 7.')
    shutil.rmtree(cfg_rutas.dir_png)
    print('borrados los PNG de 1024 px;', len(nombres), 'imagenes en el cache')
!df -h /kaggle/working

## Celda 9 - el motor de una corrida

Una sola función para poder lanzar tres arquitecturas sin repetir el código
tres veces. Orquesta, no calcula: entrenar, elegir el umbral, medir y escribir
el reporte son llamadas al paquete, que es donde están las pruebas.

Lo que hace por corrida, en su propia carpeta `corridas/<nombre>/`:

1. Entrena con `reanudar=True` y checkpoints en esa carpeta.
2. Avisa al cerrar cada época, **después** de escribir el checkpoint: si el
   aviso salió, esa época ya sobrevive a que se caiga la sesión. Mide además
   los minutos de cada época, que son la otra mitad de la comparación entre
   arquitecturas.
3. Elige el umbral sobre **validación** y toca **prueba una sola vez**, con el
   umbral ya congelado.
4. Escribe `reporte_metricas.json`, `reporte_metricas.md` y `curva_roc.svg`,
   con el mismo bloque `corrida` que el cuaderno de Colab para que las dos
   plataformas se puedan comparar con el mismo lector.

In [ ]:
# Celda 9 - motor de corrida
import json
import os
import time

import pandas as pd
import torch
from torch.utils.data import DataLoader

from copiloto_triaje.config import Config
from copiloto_triaje.datos.dataset import DatasetRadiografias
from copiloto_triaje.modelo.arquitectura import construir_modelo
from copiloto_triaje.modelo.entrenamiento import entrenar, evaluar_epoca
from copiloto_triaje.modelo.evaluacion import elegir_umbral, metricas_en_umbral
from copiloto_triaje.modelo.reporte import escribir_reporte_markdown, guardar_curva_roc

DISPOSITIVO = torch.device('cuda')
# Kaggle da pocos nucleos. Con el cache de 320 px descodificar es barato, asi
# que no hace falta exprimirlo: lo que mata es pasarse y quedarse sin memoria.
CARGADORES = min(4, os.cpu_count() or 2)


def entrenar_y_reportar(arquitectura, nombre_corrida=None, reanudar=True,
                        rehacer=False, **cambios):
    """Entrena una arquitectura y deja sus entregables en corridas/<nombre>.

    `rehacer=False` salta la corrida si ya tiene reporte: es lo que permite
    reejecutar la celda de las tres arquitecturas despues de que la sesion se
    corte a las 12 h sin repetir lo que ya estaba hecho.
    """
    nombre = nombre_corrida or f'kaggle_{arquitectura}_{TAM_CACHE}px'
    dir_corrida = CORRIDAS / nombre
    ruta_json = dir_corrida / 'reporte_metricas.json'
    if ruta_json.exists() and not rehacer:
        print(f'{nombre}: ya tiene reporte, se salta')
        return json.loads(ruta_json.read_text(encoding='utf-8'))

    cfg = Config(raiz_datos=TRABAJO, zip_rsna=CRUDO, arquitectura=arquitectura,
                 tam_imagen=TAM_CACHE, **cambios)
    # El cache esta escrito a TAM_CACHE. Entrenar a otro tamano desde el
    # redimensionaria dos veces y las metricas no corresponderian a lo que
    # describe el reporte; como el fallo es invisible, se corta aqui.
    assert cfg.tam_imagen == TAM_CACHE, (
        f'el cache es de {TAM_CACHE} px y se pidieron {cfg.tam_imagen}: '
        f'regenera el cache a ese tamano (celda 7) antes de entrenar')

    dir_checkpoints = dir_corrida / 'checkpoints'
    dir_checkpoints.mkdir(parents=True, exist_ok=True)

    def cargador(particion, entrenamiento):
        df = pd.read_csv(cfg.dir_manifiestos / f'{particion}.csv')
        # PNG320 y no cfg.dir_png: cfg.dir_png apunta a los originales de
        # 1024 px, que la celda 8 borra. Se pasa explicito porque el error
        # contrario -entrenar leyendo otra carpeta- no daria ningun sintoma.
        ds = DatasetRadiografias(df, PNG320, cfg.tam_imagen, entrenamiento)
        return DataLoader(ds, batch_size=cfg.lote, shuffle=entrenamiento,
                          num_workers=CARGADORES, pin_memory=True,
                          # Los cargadores viven las 24 epocas: sin esto los
                          # procesos de carga se crean y destruyen en cada una.
                          persistent_workers=CARGADORES > 0)

    modelo = construir_modelo(cfg.arquitectura, congelar_backbone=True,
                              preentrenado=True)

    # entrenar() no imprime nada y no mide tiempo. Lo primero deja horas de
    # silencio en las que no se distingue "avanzando" de "colgado"; lo segundo
    # deja la comparacion coja, porque elegir arquitectura es elegir entre
    # calidad y coste. flush=True porque Kaggle retiene la salida si no.
    reloj = [time.time()]

    def avisar(r):
        ahora = time.time()
        minutos = (ahora - reloj[0]) / 60
        reloj[0] = ahora
        print(f"[{cfg.arquitectura}/{r['fase']}] epoca {r['epoca']:>2} "
              f"| {minutos:5.1f} min | perdida {r['perdida']:.4f} "
              f"| recall {r['recall']:.3f} | especificidad {r['especificidad']:.3f} "
              f"| AUC {r['auc']:.3f} | umbral {r['umbral']:.3f}"
              + ('  <- mejor' if r['mejora']
                 else f"  (sin mejorar {r['sin_mejora']}/{r['paciencia']})"),
              flush=True)

    inicio = time.time()
    print(f'--- {nombre} | lote {cfg.lote} | '
          f'{cfg.epocas_cabeza}+{cfg.epocas_ajuste} epocas | '
          f'reanudar={reanudar and (dir_checkpoints / "ultimo.pt").exists()}',
          flush=True)
    resultado = entrenar(modelo, cargador('train', True), cargador('val', False),
                         cfg, DISPOSITIVO, dir_checkpoints, reanudar=reanudar,
                         al_terminar_epoca=avisar)
    segundos = time.time() - inicio

    # `modelo` vuelve de entrenar() restaurado a los pesos de su MEJOR epoca,
    # no a los de la ultima: no hace falta recargar el checkpoint a mano.
    y_val, s_val = evaluar_epoca(modelo, cargador('val', False), DISPOSITIVO)
    umbral, m_val, paso = elegir_umbral(
        y_val, s_val, cfg.especificidad_objetivo, cfg.especificidad_minima,
        cfg.recall_objetivo, cfg.recall_minimo)
    y_test, s_test = evaluar_epoca(modelo, cargador('test', False), DISPOSITIVO)
    m_test = metricas_en_umbral(y_test, s_test, umbral)

    # El bloque 'corrida' repite las claves del cuaderno de Colab -sin el, dos
    # reportes del mismo formato no dicen con que se entreno cada uno- y anade
    # plataforma y segundos, que es lo que falta para comparar coste.
    ruta_json.write_text(json.dumps(
        {'corrida': {'nombre': nombre,
                     'plataforma': 'kaggle',
                     'arquitectura': cfg.arquitectura,
                     'tam_imagen': cfg.tam_imagen,
                     'lote': cfg.lote,
                     'epocas': [cfg.epocas_cabeza, cfg.epocas_ajuste],
                     'ajuste_fino_completo': cfg.descongelar_todo,
                     'scheduler_coseno': cfg.usar_scheduler,
                     'paciencia': cfg.paciencia,
                     'edad': [cfg.edad_minima, cfg.edad_maxima],
                     'n_val': int(len(y_val)), 'n_test': int(len(y_test)),
                     'segundos_entrenamiento': round(segundos, 1),
                     'gpu': torch.cuda.get_device_name(0)},
         'umbral': umbral, 'regla': paso, 'validacion': m_val,
         'prueba': m_test, 'mejor_epoca': resultado['mejor_epoca'],
         'historial': resultado['historial']},
        indent=2, ensure_ascii=False), encoding='utf-8')

    # El recuento de omitidos es un entregable del spec (parrafo 8) y
    # preparar_datos ya lo dejo escrito al lado: no pasarlo era dejarlo fuera
    # del reporte teniendolo a mano.
    ruta_omitidos = TRABAJO / 'omitidos.json'
    omitidos = (json.loads(ruta_omitidos.read_text(encoding='utf-8'))
                if ruta_omitidos.exists() else None)
    ruta_roc = guardar_curva_roc(y_test, s_test, dir_corrida / 'curva_roc.svg',
                                 umbral=umbral)
    escribir_reporte_markdown(
        dir_corrida / 'reporte_metricas.md', umbral=umbral, paso=paso,
        metricas_validacion=m_val, metricas_prueba=m_test,
        historial=resultado['historial'], mejor_epoca=resultado['mejor_epoca'],
        ruta_roc=ruta_roc, omitidos=omitidos)

    print(f'{nombre} terminada en {segundos / 3600:.2f} h '
          f'| mejor epoca {resultado["mejor_epoca"]} | entregables en {dir_corrida}')
    for etiqueta, valor, minimo in (
            ('recall', m_test['recall'], cfg.recall_minimo),
            ('especificidad', m_test['especificidad'], cfg.especificidad_minima),
            ('AUC', m_test['auc'], 0.85)):
        print(f"  {'OK' if valor >= minimo else 'NO'} {etiqueta}: "
              f'{valor:.3f} (minimo {minimo})')

    # La siguiente arquitectura construye su modelo antes de que el recolector
    # pase por este: sin liberar aqui, las dos conviven en la VRAM.
    del modelo
    torch.cuda.empty_cache()
    return json.loads(ruta_json.read_text(encoding='utf-8'))

## Celda 10 - una sola arquitectura (opcional)

Esta celda **no hace nada por defecto**, y es a proposito.

La celda 11 entrena las tres de la mas barata a la mas cara, que es el orden que
protege el presupuesto: si la sesion se corta a las 12 h, te quedas con las dos
baratas medidas en vez de con la cara a medias. Pero `Save Version > Save & Run
All` ejecuta **todas** las celdas en orden, asi que si esta lanzara una corrida,
se comeria la parte cara del presupuesto antes de que la 11 llegue a empezar.

Para entrenar una sola, pon su nombre en `SOLO_ESTA` y ejecuta esta celda a mano
(no con Save & Run All). Util para reintentar la que se quedo sin memoria, o
para gastar el resto de la cuota en una sola arquitectura.

In [ ]:
# Celda 10 - una sola arquitectura, a mano
# None a proposito: ver el markdown de arriba. Con un nombre aqui, Save & Run
# All entrenaria esta ANTES que la celda 11 y el orden barata-primero, que es
# lo unico que protege el presupuesto de 12 h, dejaria de cumplirse.
SOLO_ESTA = None   # p. ej. 'densenet121', o 'efficientnet_b0'

if SOLO_ESTA is None:
    print('sin arquitectura seleccionada: sigue a la celda 11')
else:
    entrenar_y_reportar(SOLO_ESTA)

## Celda 11 - las tres arquitecturas, para elegir con datos

Las tres con la misma configuración y la misma partición: es la única forma de
que la comparación signifique algo. Van **de la más barata a la más cara**, así
que si la sesión se corta te quedan las baratas medidas en vez de la cara a
medias.

> ### ⚠️ GUARDA DESPUÉS DE CADA ARQUITECTURA
>
> **En Kaggle no existe nada equivalente a Drive.** `/kaggle/working` vive solo
> mientras dure la sesión: si se recicla —por inactividad, por reinicio o
> porque se agota el tiempo— **se borra entero**, con los pesos, los reportes y
> el caché de 320 px dentro.
>
> El 23-sep-2026 se perdieron así las tres corridas ya terminadas. Lo único que
> sobrevivió fueron los números, porque estaban impresos en la salida.
>
> **Save Version es el único guardado que existe aquí.** Haz *Save Version →
> Quick Save* en cuanto termine cada arquitectura, sin esperar a las tres. Y si
> vas justo de cuota, entrena de una en una con la celda 10 (`SOLO_ESTA`),
> guardando entre cada una.
>
> Como red de seguridad, esta celda imprime las métricas de cada corrida en
> cuanto acaba: mientras la salida quede en el cuaderno, los números se pueden
> recuperar aunque se pierdan los archivos.

In [ ]:
# Celda 11 - las tres en secuencia
# De la mas barata a la mas cara. Si la sesion se corta a las 12 h, quedan las
# dos baratas MEDIDAS en vez de la cara a medias: una comparacion incompleta
# sigue sirviendo, una corrida a medias no sirve para nada.
ARQUITECTURAS_A_COMPARAR = ('mobilenet_v3_large', 'efficientnet_b0', 'densenet121')

for arquitectura in ARQUITECTURAS_A_COMPARAR:
    try:
        entrenar_y_reportar(arquitectura)
    except RuntimeError as exc:
        # Un 'CUDA out of memory' en una arquitectura no debe tirar las otras
        # dos: se anota y se sigue. Cualquier otro fallo se propaga, que para
        # eso es un error.
        if 'out of memory' not in str(exc).lower():
            raise
        print(f'{arquitectura}: sin memoria en la GPU. Reintenta esa sola con '
              f"entrenar_y_reportar('{arquitectura}', lote=16)", flush=True)
        fallo_sin_memoria = True
    else:
        fallo_sin_memoria = False
    if fallo_sin_memoria:
        # empty_cache() FUERA del except. Dentro, exc.__traceback__ mantiene
        # vivos los frames de entrenar_y_reportar y con ellos el modelo en la
        # GPU y el estado de Adam; empty_cache solo devuelve bloques que ya no
        # tienen referencias, asi que ahi no liberaba lo que ocupaba la corrida
        # que acababa de fallar, y la siguiente -mas pesada- arrancaba con la
        # memoria todavia ocupada.
        torch.cuda.empty_cache()

## Celda 12 - cuál gana

No se decide aquí. `scripts/comparar_corridas.py` lee los
`reporte_metricas.json` de todas las carpetas de `corridas/` -incluidas las de
sesiones anteriores, si se adjuntó su salida- y escribe la comparativa con la
regla de selección dentro del propio informe. Para eso el bloque `corrida` del
JSON tiene las mismas claves que el del cuaderno de Colab: sin él, dos reportes
del mismo formato no dicen con qué se entrenó cada uno.

La regla, en corto: gana la que satisface más criterios del spec (recall >=
0,80, especificidad >= 0,70, AUC >= 0,85); a igualdad, mayor recall, porque el
falso negativo es el error caro; si empatan también en recall, mayor AUC; y si
ninguna cumple los tres mínimos, no se corona ninguna.

Vive en el script y no en este cuaderno por lo mismo que el resto de la lógica:
elegir modelo es justo la decisión que hay que defender en el informe, y tiene
que salir de un archivo con pruebas, igual en Kaggle que en local.

Para meter en la comparativa la **corrida 1** (Colab, mobilenet_v3_large a
224 px: recall 0,766, especificidad 0,764, AUC 0,845), copia su
`reporte_metricas.json` a `corridas/corrida1_colab/` y vuelve a ejecutar.

In [ ]:
# Celda 12 - comparar con el script del repo, no con una tabla escrita aqui
import pathlib
import subprocess
import sys

# Rutas recalculadas, no heredadas de la celda 2: un reinicio de sesion borra
# las variables de Python, y esta celda reventaba con NameError justo al final,
# con las tres corridas ya hechas.
CODIGO = pathlib.Path('/kaggle/working/codigo')
CORRIDAS = pathlib.Path('/kaggle/working/corridas')
comparador = CODIGO / 'scripts' / 'comparar_corridas.py'
assert comparador.exists(), (
    'El Dataset del codigo es anterior a scripts/comparar_corridas.py. Sube '
    'una New Version de ese Dataset con el repo al dia y vuelve a adjuntarlo.')

carpetas = sorted(ruta.parent for ruta in CORRIDAS.glob('*/reporte_metricas.json'))
if not carpetas:
    print('todavia no hay ninguna corrida con reporte en', CORRIDAS)
else:
    ruta_comparativa = CORRIDAS / 'comparativa.md'
    subprocess.run([sys.executable, str(comparador), *map(str, carpetas),
                    '--salida', str(ruta_comparativa)], check=True)
    # Se imprime ademas de escribirlo: el Markdown es el entregable que se
    # baja, pero en una corrida en batch (Save & Run All) la salida de la
    # celda es lo unico que se mira.
    print()
    print(ruta_comparativa.read_text(encoding='utf-8'))

## Al terminar

1. **Save Version > Save & Run All (Commit)**. Corre el cuaderno entero en
   segundo plano (la sesión interactiva se cierra sola; esta no) y deja
   `/kaggle/working` como Dataset de salida de esa versión.
   *Quick Save* guarda el cuaderno con las salidas que ya hay, sin reejecutar:
   sirve para no perder el trabajo, no para entrenar.
2. En la sesión siguiente: **+ Add Input > Your Work / Notebook Output**,
   elegir esta versión. La celda 4 encuentra sola `png320/`,
   `trabajo/manifiestos/` y `corridas/`, y se salta la conversión.
3. **Lo que hay que bajarse** de `corridas/<nombre>/`:
   - `checkpoints/pesos_modelo_cpu.pt`: los pesos limpios en CPU, el entregable
     para SP2. Los `mejor.pt` y `ultimo.pt` no sirven ahí: mezclan pesos con
     estado del optimizador y sus tensores están en CUDA.
   - `reporte_metricas.json`, `reporte_metricas.md` y `curva_roc.svg`.
   - de `corridas/`, la `comparativa.md` que escribe la celda 12: es la que
     dice cuál gana y con qué regla.
4. El consumo de GPU de la semana está en el panel derecho de la sesión.

**El umbral se eligió en validación y prueba se tocó una sola vez.** Volver a
evaluar sobre prueba para "mejorar" el número invalida el reporte, y en ese
caso hay que declararlo.